# 1.1 — Process review

Applies `PROCESS_REVIEW` from [ai_lca_config.py](ai_lca_config.py) on top of the
AI-proposed process structure from notebook 1: include/exclude, rename,
merge-into, or re-parent a process. Process IDs are stable — a merge
reattaches that process's flows to the target process rather than dropping
them, and removing a process without a merge drops its attached flows.

Only processes listed in `PROCESS_REVIEW` change; everything else keeps the
AI proposal exactly as extracted.

In [ ]:
# Moved into paper_reading_notebooks/ — walk up to the project root (marked by
# dashboard_config.py) and chdir there so ai_lca_config/ai_lca imports and every
# relative path below (ai_lca_outputs/, SOURCE_DOCUMENT_PATHS, ...) resolve exactly
# as they did when this notebook lived at the project root.
import os
import sys
from pathlib import Path

_here = Path.cwd()
for _candidate in [_here, *_here.parents]:
    if (_candidate / "dashboard_config.py").exists():
        _project_root = _candidate
        break
else:
    raise RuntimeError("Could not locate project root (dashboard_config.py) from cwd: " + str(_here))

os.chdir(_project_root)
if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))
print("Project root:", _project_root)


In [ ]:
import ai_lca_config as cfg
cfg.print_config()

from ai_lca.export import extraction_to_dataframe, process_structure_to_dataframe
from ai_lca.review import apply_process_review, process_review_id_map, remap_inventory_dataframe
from ai_lca.notebook_helpers import load_extraction, process_review_dataframe, run_output_dir, save_extraction

run_dir = run_output_dir(cfg.OUTPUT_DIR, cfg.RUN_LABEL)
raw_path = run_dir / "1_extraction_raw.json"
if not raw_path.exists():
    raise FileNotFoundError(f"{raw_path} not found — run 1.paper_ingest_and_extract.ipynb first.")

extraction = load_extraction(raw_path)
print(f"Loaded extraction from {raw_path}")
print()
print("Process structure BEFORE review:")
process_structure_to_dataframe(extraction)

## Apply `PROCESS_REVIEW`

In [ ]:
review_df = process_review_dataframe(extraction, cfg.PROCESS_REVIEW)
reviewed_extraction = apply_process_review(extraction, review_df)

new_warnings = [w for w in reviewed_extraction.assumptions_or_warnings
                if w not in extraction.assumptions_or_warnings]
if new_warnings:
    print("Changes applied:")
    for w in new_warnings:
        print("  -", w)
else:
    print("No PROCESS_REVIEW overrides were set — process structure unchanged.")

print()
print(f"Processes: {len(extraction.processes)} -> {len(reviewed_extraction.processes)}")

### Process structure AFTER review

In [ ]:
process_structure_to_dataframe(reviewed_extraction)

## Remap the flow list onto the reviewed process structure

Flows attached to a removed/merged process are reassigned (or dropped) to
match, using the same stable `flow_id`s from notebook 1.

In [ ]:
id_map = process_review_id_map(extraction, review_df)
process_names = {p.process_id: p.name for p in reviewed_extraction.processes}
original_inventory_df = extraction_to_dataframe(extraction)
inventory_df = remap_inventory_dataframe(
    original_inventory_df, process_id_map=id_map, process_names=process_names,
)
print(f"Flows: {len(original_inventory_df)} -> {len(inventory_df)}")
inventory_df

## Save reviewed extraction + remapped inventory for the next notebook

In [ ]:
extraction_path = save_extraction(reviewed_extraction, run_dir / "1_1_extraction_reviewed.json")
inventory_path = run_dir / "1_1_inventory_after_process_review.csv"
inventory_df.to_csv(inventory_path, index=False)
print("Saved reviewed extraction to:", extraction_path)
print("Saved remapped inventory to: ", inventory_path)
print()
print("Next: open ai_lca_config.py, fill in INVENTORY_REVIEW using the flow_id values")
print("above, save, then run 1.2.paper_inventory_review.ipynb.")